## Сервис для мониторинга своевременной оплаты аренды гаража

В данной тетрадке описана логика, которая: 

 1. Читает график арендных платежей (arenda.xlsx) с информацией о номере гаража, сумме аренды и дате начала аренды.
    
 2. Генерирует расписание ожидаемых платежей для каждого гаража по месяцам до текущего периода.

 3. Читает выписку банка (print 2.xlsx) и парсит данные о дате и сумме операции, сумму операции сопоставляет с таблицей аренды, чтобы идентифицировать гараж

 4. Сопоставляет ожидаемые и фактические платежи по месяцу и сумме.

 5. Определяет статус каждого платежа:
     - "получен": платеж найден в банке.
     - "срок не наступил": ещё не истек период 3 дня после due_date.
     - "просрочен": прошло более 3 дней после due_date и платеж не найден.
       
 6. При необходимости может отправлять уведомления в Telegram-бот о просроченных платежах и трёх подряд просрочках (чтобы можно было взять арендатора на контроль, либо прекратить с ним работу).

 7. Экспортирует итоговый отчёт в Excel: названия гаражей, ожидаемая и фактическая дата, сумма, статус.


In [ ]:
# Импорт необходимых библиотек

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import calendar
import requests
import re  

# Путь к файлу arenda
ARRENDA_PATH = "/kaggle/input/897455/arenda.xlsx"
# Путь к файлу банковской выписки
BANK_PATH   = "/kaggle/input/897455/print 2.xlsx"

# Данные бота Telegram, в который будут посылаться уведомления
TELEGRAM_TOKEN   = "YOUR_BOT_TOKEN"
TELEGRAM_CHAT_ID = "YOUR_CHAT_ID"

In [ ]:
# Определение функций, которые нам понадобятся в работе:

def read_arenda(path):
    """
    Читает Excel-файл с графиком аренд.
    Переименовывает в [garage, amount, start_date].
    Приводит типы: float для суммы, datetime для даты.
    Возвращает DataFrame с колонками ['garage','amount','start_date'].
    """
    df = pd.read_excel(path)
    # Стандартизируем имена столбцов
    df.columns = df.columns.str.strip().str.lower()
    df.rename(columns={
        'гараж': 'garage',
        'сумма': 'amount',
        'первоначальная дата': 'start_date'
    }, inplace=True)
    # Приводим типы
    df['amount']     = pd.to_numeric(df['amount'], errors='coerce').round(2)
    df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
    return df[['garage','amount','start_date']]


def read_bank(path):
    """
    Построчно парсит банковскую выписку.
    Ищет строку, содержащую 'Дата операции', чтобы определить заголовок.
    Читает все операции ниже этой строки до конца файла.
    Парсит дату и сумму платежа.
    Возвращает DataFrame с ['date','amount','period'].
    """
    raw = pd.read_excel(path, header=None)
    header_row = date_idx = sum_idx = None
    for idx, row in raw.iterrows():
        cells = row.astype(str).str.lower()
        text = ' '.join(cells.dropna())
        if 'дата операции' in text:
            header_row = idx
            # Находим позиции
            for i, cell in enumerate(cells):
                if 'дата операции' in cell:
                    date_idx = i
                if 'сумма в валюте счёта' in cell:
                    sum_idx = i
            break
    if header_row is None or date_idx is None or sum_idx is None:
        raise ValueError("Не найдена строка с заголовками в банковской выписке.")
    # Собираем операции
    data = []
    for j in range(header_row+1, len(raw)):
        row = raw.iloc[j]
        date_cell = str(row[date_idx])
        sum_cell  = str(row[sum_idx])
        # Пропускаем пустые или некорректные строки
        if not re.search(r"\d{2}[./]\d{2}[./]\d{4}", date_cell):
            continue
        # Извлекаем дату
        dm = re.search(r"(\d{2}[./]\d{2}[./]\d{4})", date_cell)
        dt = pd.to_datetime(dm.group(1), dayfirst=True, errors='coerce')
        # Извлекаем сумму
        sm = re.search(r"([+-]?\d{1,3}(?:[ \u202f]?\d{3})*[.,]\d{2})", sum_cell)
        if not sm or pd.isna(dt):
            continue
        amt = float(sm.group(1).replace('+','').replace(' ','')\
                        .replace('\u202f','').replace(',','.'))
        data.append({'date': dt, 'amount': round(amt,2)})
    bank_df = pd.DataFrame(data)
    if bank_df.empty:
        raise ValueError("Выписка банка не содержит операций.")
    # Добавляем период для объединения
    bank_df['period'] = bank_df['date'].dt.to_period('M')
    return bank_df


def adjust_due_date(dt):
    """
    Функция корректирует дату оплаты:
    Если день > числа дней месяца, возвращает последний день (для месяцев с 30,29, 28 днями).
    """
    y, m = dt.year, dt.month
    last = calendar.monthrange(y, m)[1]
    return datetime(y, m, min(dt.day, last))


def build_schedule(arenda_df):
    """
    Формирует расписание ожидаемых платежей по каждому гаражу:
    От первоначальной даты (start_date) до текущего месяца.
    Для каждого месяца вычисляет due_date (последний рабочий день).
    Возвращает DataFrame с ['garage','due_date','amount','period'].
    """
    schedules = []
    current = pd.Timestamp.today().to_period('M')
    for _, row in arenda_df.iterrows():
        start = row['start_date'].to_period('M')
        for per in pd.period_range(start, current, freq='M'):
            due = adjust_due_date(per.to_timestamp())
            schedules.append({
                'garage':   row['garage'],
                'due_date': due.date(),
                'amount':   row['amount'],
                'period':   per
            })
    return pd.DataFrame(schedules)


def build_status(arenda_df, bank_df):
    """
    Сопоставляет график и выписку:
      - 'получен'    — если найдена транзакция matching period & amount.
      - 'срок не наступил' — если еще не истекло 3 дня после due_date.
      - 'просрочен'  — иначе.
    Возвращает итоговый DataFrame.
    """
    schedule = build_schedule(arenda_df)
    merged   = pd.merge(
        schedule,
        bank_df[['period','amount','date']],
        on=['period','amount'], how='left'
    )
    today = pd.Timestamp.today().normalize()
    records = []
    for _, row in merged.iterrows():
        paid   = pd.notna(row['date'])
        actual = row['date'].date() if paid else None
        due_ts = pd.Timestamp(row['due_date'])
        if paid:
            status = 'получен'
        else:
            status = 'просрочен' if today > due_ts + pd.Timedelta(days=3) else 'срок не наступил'
        records.append({
            'Название гаража':          row['garage'],
            'Ожидаемая дата оплаты':    row['due_date'],
            'Фактическая дата оплаты':   actual,
            'Сумма оплаты':             row['amount'],
            'Статус':                    status
        })
    return pd.DataFrame(records)

In [ ]:
#Телеграм блок. Функции


def send_telegram_alerts(status_df):
    """
    Отправляет в Telegram сообщения о просроченных платежах.
    """
    if TELEGRAM_TOKEN == "YOUR_BOT_TOKEN":
        print("Укажите TELEGRAM_TOKEN и TELEGRAM_CHAT_ID.")
        return
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    for _, r in status_df[status_df['Статус']=='просрочен'].iterrows():
        text = (
            f"🚨 Просрочен платеж:\n"
            f"{r['Название гаража']}\n"
            f"{r['Сумма оплаты']}\n"
            f"ожидалось {r['Ожидаемая дата оплаты']}\n"
            f"не найдено до { (pd.Timestamp(r['Ожидаемая дата оплаты']) + pd.Timedelta(days=3)).date() }"
        )
        requests.post(url, data={'chat_id': TELEGRAM_CHAT_ID, 'text': text})


def add_consecutive_late_alerts(status_df):
    """
    Отправляет Telegram предупреждение для гаражей с 3 просроченными платежами подряд.
    """
    alerts = []
    for garage, grp in status_df.groupby('Название гаража'):
        cnt = 0
        for st in grp.sort_values('Ожидаемая дата оплаты')['Статус']:
            cnt = cnt + 1 if st=='просрочен' else 0
            if cnt >= 3:
                alerts.append(garage)
                break
    for g in set(alerts):
        requests.post(
            f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage",
            data={'chat_id': TELEGRAM_CHAT_ID,
                  'text': f"🔴 Гараж '{g}' просрочил 3 месяца подряд. Взять на контроль."}
        )

In [ ]:
def export_report(status_df):
    """
    Сохраняет итоговый отчёт в Excel по шаблону garage_payment_status_YYYY-MM-DD.xlsx.
    """
    today = datetime.today().strftime('%Y-%m-%d')
    path  = f"garage_payment_status_{today}.xlsx"
    status_df.to_excel(path, index=False)
    print(f"✅ Отчёт сохранён: {path}")

Понимаю, что в требовании этого не было, но я решила, что в отчете хочется отразить фактическую дату платежа из выписки и ожидаемую.

In [ ]:
if __name__ == '__main__':
    # Основной поток: читаем данные, строим отчёт, уведомляем, экспортируем.
    arenda = read_arenda(ARRENDA_PATH)
    bank   = read_bank(BANK_PATH)
    status = build_status(arenda, bank)
    print(status)
    send_telegram_alerts(status)
    add_consecutive_late_alerts(status)
    export_report(status)